# Pipeline
Este codigo sirve para ejecutar en un solo recorrido todo el proceso del estudio de morfado de imagenes.

In [ ]:
import glob
import cv2 
from skimage.feature import local_binary_pattern
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from skimage.feature import hog
import numpy as np
import pickle
from scipy.signal import correlate2d
from sklearn.model_selection import LeaveOneOut
from sklearn.svm import SVC
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import os
import gc

## Carga de datasets

Los dataset tienen que estar listos cuando se ejecuten los códigos. Además de que se ejecuta por dataset el archivo entero.

In [ ]:
images_bonafide = glob.glob(f'/home/branco/Documentos/Investigacion/scripts/Caras_seleccionadas/FEI/*.jpg')
images_morphing = glob.glob(f'/home/branco/Documentos/Investigacion/scripts/Morphing/face-morphing/OpenCV_Morphing_FEI/*.jpg')

all_rutes = images_bonafide + images_morphing # Unir las rutas

y = np.hstack((np.zeros(len(images_bonafide)), np.ones(len(images_morphing)))) # Etiquetar las imagenes, 0 Bonafide y 1 Morphing

## Extracción de características

In [ ]:
from HOG import HOG_features
from uLBP import uLBP_features
from BSIF import bsif_features

vec_hog = []
vec_ulbp = []
vec_bsif = []

for rute in all_rutes:
    img = cv2.imread(rute)
    if img is None:
        print(f'Error al leer {rute}')
        continue
    
    canal = img[:, :, 2] # 0 azul, 1 verde, 2 Rojo
    vec_hog.append(HOG_features(canal))
    vec_ulbp.append(uLBP_features(canal))
    vec_bsif.append(bsif_features(canal, texture='/home/branco/Documentos/Investigacion/scripts/Pipeline/ICAtextureFilters_5x5_9bit.mat'))
    
vec_hog = np.array(vec_hog)
vec_ulbp = np.array(vec_ulbp)
vec_bsif = np.array(vec_bsif)

print(f"HOG original: {vec_hog.shape}")
print(f"uLBP original: {vec_ulbp.shape}")
print(f"BSIF original: {vec_bsif.shape}")

## Seleción de características

In [ ]:
from Feature_selection import obtener_indices_ganadores
# HOG 1000 features
# uLBP 400 features
# BSIF 5x5 9 bits
indices_hog = obtener_indices_ganadores(vec_hog, y, cantidad_a_retener=1000)
indices_ulbp = obtener_indices_ganadores(vec_ulbp, y, cantidad_a_retener=400) 
indices_bsif = obtener_indices_ganadores(vec_bsif, y, cantidad_a_retener=100)

hog_optimizado = indices_hog[:, indices_hog]
ulbp_optimizado = indices_ulbp[:, indices_ulbp]
bsif_optimizado = indices_bsif[:, indices_bsif]

## Entrenamiento de modelos

In [ ]:
archivo_respaldo = "checkpoint_evaluacion_HOG_SVM.pkl"

if os.path.exists(archivo_respaldo):
    with open(archivo_respaldo, "rb") as f:
        estado = pickle.load(f)
        iteraciones_completadas = estado['iteraciones_completadas']
        etiquetas_reales = estado['etiquetas']
        probabilidades_predichas = estado['probabilidades']
    print(f"Reanudando desde la imagen {iteraciones_completadas}...")
else:
    # Iniciar desde cero
    iteraciones_completadas = 0
    etiquetas_reales = []
    probabilidades_predichas = []
    print("Iniciando evaluación LOO desde cero...")
    
loo = LeaveOneOut()
total_imagenes = len(y)

for iteracion_actual, (train_index, test_index) in enumerate(loo.split(y)):
    if iteracion_actual < iteraciones_completadas:
        continue
    
    print(f"Evaluando imagen {iteracion_actual + 1} de {total_imagenes}...")
        
    y_train, y_test = y[train_index], y[test_index]
    
    # TODO Quitar comentarios para el entrenamiento con el feature extraction
    hog_train, hog_test = vec_hog[train_index], vec_hog[test_index]
    #ulbp_train, ulbp_test = vec_ulbp[train_index], vec_ulbp[test_index]
    #bsif_train, bsif_test = vec_bsif[train_index], vec_bsif[test_index]
    
    # 2. SELECCIÓN DE CARACTERÍSTICAS (¡SOLO EN TRAIN!)
    # Esto asegura que la imagen de prueba sea 100% desconocida
    idx_hog = obtener_indices_ganadores(hog_train, y_train, cantidad_a_retener=200)
    #idx_ulbp = obtener_indices_ganadores(ulbp_train, y_train, cantidad_a_retener=20)
    #idx_bsif = obtener_indices_ganadores(bsif_train, y_train, cantidad_a_retener=80)
    
    # 3. FILTRAR AMBOS CONJUNTOS CON LOS ÍNDICES DE TRAIN
    hog_train_opt, hog_test_opt = hog_train[:, idx_hog], hog_test[:, idx_hog]
    #ulbp_train_opt, ulbp_test_opt = ulbp_train[:, idx_ulbp], ulbp_test[:, idx_ulbp]
    #bsif_train_opt, bsif_test_opt = bsif_train[:, idx_bsif], bsif_test[:, idx_bsif]
    
    modelo = SVC(kernel='linear', probability=True) 
    modelo.fit(hog_train_opt, y_train)
    
    prob_morphing = modelo.predict_proba(hog_test_opt)[0, 1]
    
    probabilidades_predichas.append(prob_morphing)
    etiquetas_reales.append(y_test[0])
    
    iteraciones_completadas += 1
    del modelo, hog_train, hog_train_opt
    gc.collect()
    
    estado_actual = {
        'iteraciones_completadas': iteraciones_completadas,
        'etiquetas': etiquetas_reales,
        'probabilidades': probabilidades_predichas
    }
    
    # Se sobrescribe el archivo en cada iteración
    if iteraciones_completadas % 10 == 0 or iteraciones_completadas == total_imagenes:
        with open(archivo_respaldo, "wb") as f:
            pickle.dump(estado_actual, f)

In [ ]:
# =========================================================
# CÁLCULO DEL D-EER (Detection Equal Error Rate)
# =========================================================
# Calculamos la curva ROC (Falsos Positivos vs Verdaderos Positivos)
fpr, tpr, thresholds = roc_curve(etiquetas_reales, probabilidades_predichas)

# Calculamos Falsos Negativos (False Rejection Rate)
fnr = 1 - tpr

# El D-EER es el punto exacto donde False Acceptance Rate (FPR) == False Rejection Rate (FNR)
eer_threshold = brentq(lambda x : 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
deer = interp1d(fpr, tpr)(eer_threshold)

print(f"El D-EER del modelo es: {1 - deer:.4%}")